# ClusterBP v2: аудит утечек

Насколько каждая типичная ошибка протокола завышает качество — на одних и тех же окнах, одной модели и одних фолдах.

**Модель** — XGB_full (52 признака): по качеству она совпадает с ClusterBP v2 (`metrics_A.csv`, `metrics_B.csv`),
а обучается за минуты, поэтому все сочетания ошибок можно честно переобучить.

**Корректный протокол** (как в `clusterbp_v2.ipynb`):
разбиение по пациентам; шаблоны фенотипов только по обучающим пациентам фолда; зазор между историей и горизонтом
(A — 1 день, B — 7 дней); порог Macro F1 подобран на внутренней валидации; тест при реальной доле событий.

**Ошибки** (каждая — один переключатель):

| Ошибка | Что меняется | Переобучение |
|---|---|---|
| `случайное разбиение окон` | фолды по окнам (`StratifiedKFold`), окна одного пациента попадают и в обучение, и в тест | да |
| `фенотипы на всех пациентах` | шаблоны GDTW строятся по всем пациентам, включая тестовых | да |
| `без зазора` | горизонт начинается сразу после окна истории: `[t, t + 7)` | да |
| `порог на тесте` | порог Macro F1 подбирается на тестовом фолде | нет |
| `сбалансированный тест` | в тесте отрицательные окна прорежены до числа положительных (1 : 1); порог — на так же прореженной валидации | нет |

При случайном разбиении окон «обучающие пациенты» — почти все пациенты, поэтому там шаблоны фенотипов всегда строятся по всем.

Считаются все сочетания (полный факторный план), из него берутся:
- каждая ошибка по отдельности относительно корректного протокола;
- все ошибки вместе (≈ условия первой версии статьи);
- все ошибки, кроме одной (вклад ошибки, когда остальные уже сделаны).

95% ДИ разностей — парный бутстреп по пациентам (пациент берётся целиком со всеми окнами во всех вариантах).
Для сравнения добавлено правило «максимальный САД окна»: как меняется отрыв модели от правила.

In [ ]:
import os
import time
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

from clusterbp_lib import RESULTS_DIR, DATA_DIR, SEED, TARGETS, FEATURE_SETS, N_FOLDS, INNER_VAL_FRAC, fold_masks, best_threshold
from clusterbp_extra import (load_windows, phenotypes, event_target, fit_predict, paired_boot, point_metrics, ci, pval,
                             N_BOOT)

warnings.filterwarnings('ignore')
OUT = os.path.join(RESULTS_DIR, 'leakage')
os.makedirs(OUT, exist_ok=True)
AUDIT_TARGETS = ['A', 'B']
FULL = FEATURE_SETS['full']

win, pheno_win, pheno_first, pheno_first_pid = load_windows()
events = pd.read_parquet(os.path.join(DATA_DIR, 'events.parquet'))
pid = win['pid'].values
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_rand = np.full(len(win), -1)
for k, (_, te) in enumerate(skf.split(win, win['y_A'])):
    fold_rand[te] = k
FOLDS = {'patient': win['fold'].values, 'random': fold_rand}

Y = {}
for t in AUDIT_TARGETS:
    cfg = TARGETS[t]
    Y[(t, 'gap')] = win[f'y_{t}'].values
    Y[(t, 'nogap')] = event_target(win, events, cfg['codes'], 0, cfg['horizon'])
pd.DataFrame({f'{t}, {g}': [Y[(t, g)].mean()] for (t, g) in Y}, index=['доля положительных окон'])

In [ ]:
def masks(split, k):
    """Обучение / внутренняя валидация / тест для фолда k."""
    if split == 'patient':
        return fold_masks(win, k)
    te = FOLDS['random'] == k
    rng = np.random.default_rng(SEED + k)
    va = ~te & (rng.random(len(win)) < INNER_VAL_FRAC)
    return ~te & ~va, va, te


def balanced_subset(y, mask, seed):
    """Маска 1 : 1 внутри mask: прореживается больший класс."""
    idx_pos, idx_neg = np.flatnonzero(mask & (y == 1)), np.flatnonzero(mask & (y == 0))
    rng = np.random.default_rng(seed)
    if len(idx_pos) <= len(idx_neg):
        keep = np.concatenate([idx_pos, rng.choice(idx_neg, len(idx_pos), replace=False)])
    else:
        keep = np.concatenate([rng.choice(idx_pos, len(idx_neg), replace=False), idx_neg])
    out = np.zeros(len(y), bool)
    out[keep] = True
    return out


# Конфигурации обучения: (разбиение, фенотипы, зазор)
TRAIN_CFGS = [('patient', 'train', 'gap'), ('patient', 'train', 'nogap'), ('patient', 'all', 'gap'),
              ('patient', 'all', 'nogap'), ('random', 'all', 'gap'), ('random', 'all', 'nogap')]
runs = {}       # (t, split, pheno, gap) -> {'score', 'thr_val', 'thr_val_bal'}
rule_thr = {}   # (t, gap) -> пороги правила по фолдам
t_all = time.time()
for split in ['patient', 'random']:
    for k in range(N_FOLDS):
        tr, va, te = masks(split, k)
        for pheno in ['train', 'all']:
            if (split, pheno, 'gap') not in TRAIN_CFGS:
                continue
            phenotypes(win, pheno_win, pheno_first, pheno_first_pid, k, all_patients=(pheno == 'all'))
            X = win[FULL].values.astype(float)
            for t in AUDIT_TARGETS:
                for gap in ['gap', 'nogap']:
                    y = Y[(t, gap)]
                    s, thr, _ = fit_predict(win, FULL, y, tr, va, SEED + k, X=X)
                    vb = balanced_subset(y, va, SEED + 100 + k)
                    thr_b, _ = best_threshold(y[vb], s[vb])
                    r = runs.setdefault((t, split, pheno, gap), {'score': np.full(len(win), np.nan),
                                                                 'thr_val': {}, 'thr_val_bal': {}})
                    r['score'][te] = s[te]
                    r['thr_val'][k] = thr
                    r['thr_val_bal'][k] = thr_b
                    if split == 'patient' and pheno == 'train':
                        sr = win['sbp_max'].values
                        rule_thr[(t, gap, k)] = (best_threshold(y[va], sr[va])[0], best_threshold(y[vb], sr[vb])[0])
        print(f'{split}, фолд {k}: {(time.time() - t_all) / 60:.1f} мин')

In [ ]:
ERRORS = ['случайное разбиение окон', 'фенотипы на всех пациентах', 'без зазора', 'порог на тесте', 'сбалансированный тест']


def evaluate(t, err, model='XGB_full'):
    """(mask, y, score, pred) для набора ошибок err (множество названий из ERRORS)."""
    split = 'random' if ERRORS[0] in err else 'patient'
    pheno = 'all' if (ERRORS[1] in err or split == 'random') else 'train'
    gap = 'nogap' if ERRORS[2] in err else 'gap'
    on_test, bal = ERRORS[3] in err, ERRORS[4] in err
    y = Y[(t, gap)]
    fold = FOLDS[split]
    if model == 'XGB_full':
        r = runs[(t, split, pheno, gap)]
        s = r['score']
    else:
        s = win['sbp_max'].values.astype(float)
    mask = balanced_subset(y, np.ones(len(y), bool), SEED + 7) if bal else np.ones(len(y), bool)
    pred = np.zeros(len(y), np.int8)
    for k in range(N_FOLDS):
        rows = mask & (fold == k)
        if on_test:
            thr = best_threshold(y[rows], s[rows])[0]
        elif model == 'XGB_full':
            thr = r['thr_val_bal' if bal else 'thr_val'][k]
        else:
            thr = rule_thr[(t, gap, k)][1 if bal else 0]
        pred[rows] = s[rows] > thr
    return mask, y, s, pred


ALL_SETS = [frozenset(c) for n in range(len(ERRORS) + 1) for c in combinations(ERRORS, n)]
single, loo, factorial, rule_rows = [], [], [], []
for t in AUDIT_TARGETS:
    t0 = time.time()
    sets = ALL_SETS
    entries = [evaluate(t, e) for e in sets]
    rule_sets = [frozenset(), frozenset(ERRORS)]
    entries += [evaluate(t, e, model='rule_maxSBP') for e in rule_sets]
    B = paired_boot(entries, pid)
    pm = [point_metrics(m_[1][m_[0]], m_[2][m_[0]], m_[3][m_[0]]) for m_ in entries]
    ref = sets.index(frozenset())
    full_err = sets.index(frozenset(ERRORS))

    def delta(j, jr, label, table):
        row = {'target': t, 'вариант': label, 'AUPRC': pm[j]['AUPRC'], 'macro_F1': pm[j]['macro_F1'],
               'AUROC': pm[j]['AUROC'], 'доля событий в тесте': pm[j]['доля событий']}
        for mi, m in [(1, 'AUPRC'), (2, 'macro_F1'), (0, 'AUROC')]:
            d = B[:, j, mi] - B[:, jr, mi]
            row[f'Δ{m}'] = pm[j][m] - pm[jr][m]
            row[f'Δ{m} 95% ДИ'] = ci(d)
            row[f'p Δ{m}'] = pval(d)
        table.append(row)

    single.append({'target': t, 'вариант': 'корректный протокол', 'AUPRC': pm[ref]['AUPRC'], 'macro_F1': pm[ref]['macro_F1'],
                   'AUROC': pm[ref]['AUROC'], 'доля событий в тесте': pm[ref]['доля событий']})
    for e in ERRORS:
        delta(sets.index(frozenset([e])), ref, f'+ {e}', single)
    delta(full_err, ref, 'все ошибки вместе', single)
    for e in ERRORS:
        # вклад ошибки e, когда остальные уже сделаны: (все ошибки) − (все, кроме e)
        j_wo = sets.index(frozenset(ERRORS) - {e})
        delta(full_err, j_wo, f'вклад «{e}» поверх остальных', loo)
    for j, e in enumerate(sets):
        factorial.append({'target': t, **{er: int(er in e) for er in ERRORS}, **{m: pm[j][m] for m in
                          ['AUROC', 'AUPRC', 'macro_F1', 'sensitivity', 'specificity', 'PPV', 'доля событий']}})
    for label, jm, jr in [('корректный протокол', ref, len(sets)), ('все ошибки вместе', full_err, len(sets) + 1)]:
        row = {'target': t, 'условия': label, 'AUPRC модели': pm[jm]['AUPRC'], 'AUPRC правила': pm[jr]['AUPRC'],
               'macro_F1 модели': pm[jm]['macro_F1'], 'macro_F1 правила': pm[jr]['macro_F1']}
        for mi, m in [(1, 'AUPRC'), (2, 'macro_F1')]:
            d = B[:, jm, mi] - B[:, jr, mi]
            row[f'отрыв модели от правила: Δ{m}'] = pm[jm][m] - pm[jr][m]
            row[f'Δ{m} 95% ДИ'] = ci(d)
        rule_rows.append(row)
    print(f'{t}: бутстреп {time.time() - t0:.0f} с')

single_df, loo_df = pd.DataFrame(single), pd.DataFrame(loo)
fact_df, rule_df = pd.DataFrame(factorial), pd.DataFrame(rule_rows)
for name, df in [('leakage_single', single_df), ('leakage_on_top', loo_df), ('leakage_factorial', fact_df),
                 ('leakage_vs_rule', rule_df)]:
    df.to_csv(os.path.join(OUT, f'{name}.csv'), index=False)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
display(single_df.round(3))

## Вклад ошибки, когда остальные уже сделаны

Ошибки взаимодействуют: например, порог на тесте на сбалансированной выборке даёт другой прирост, чем при реальной доле событий.
Здесь — разность «все ошибки» минус «все ошибки, кроме этой».

In [ ]:
display(loo_df.round(3))
display(rule_df.round(3))

## Итоговая таблица «ошибка → насколько завышает»

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric in zip(axes, ['AUPRC', 'macro_F1']):
    sub = single_df[single_df['вариант'] != 'корректный протокол']
    labels = sub['вариант'].unique()
    x = np.arange(len(labels))
    for i, t in enumerate(AUDIT_TARGETS):
        s_ = sub[sub.target == t].set_index('вариант').loc[labels]
        lo_hi = np.array([[float(v) for v in c.strip('[]').split(';')] if c else [np.nan, np.nan]
                          for c in s_[f'Δ{metric} 95% ДИ']])
        val = s_[f'Δ{metric}'].values
        ax.bar(x + (i - 0.5) * 0.38, val, 0.38, label=t,
               yerr=np.abs(np.vstack([val - lo_hi[:, 0], lo_hi[:, 1] - val])), capsize=3)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace('+ ', '') for l in labels], rotation=25, ha='right')
    ax.set_ylabel(f'Δ{metric} относительно корректного протокола')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'leakage_single.png'), dpi=200, bbox_inches='tight')
plt.show()

summary = single_df.pivot_table(index='вариант', columns='target', values=['ΔAUPRC', 'Δmacro_F1'], sort=False)
summary.to_csv(os.path.join(OUT, 'leakage_summary.csv'))
display(summary.round(3))